In [67]:
import time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from scipy import ndimage

SEED = 42                 # used for the data split 
SEEDS = [42, 43, 44]      # three training seeds (for weight , augmentation and training images order)
EPOCHS = 60               
BATCH_SIZE = 32
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer","dog", "frog", "horse", "ship", "truck"]

In [68]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    if "data_batch_1" in files:
        print("FOUND CIFAR-10 HERE:", root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/pankrzysiu
/kaggle/input/datasets/pankrzysiu/cifar10-python
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py
FOUND CIFAR-10 HERE: /kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py


In [69]:
DATA_DIR = "/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py"
import os

print(os.listdir(DATA_DIR))

['data_batch_1', 'data_batch_2', 'batches.meta', 'test_batch', 'data_batch_3', 'data_batch_5', 'data_batch_4', 'readme.html']


In [70]:
import os
import pickle
import numpy as np

DATA_DIR = "/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py"

def load_batch(file_path):
    with open(file_path, "rb") as f:
        batch = pickle.load(f, encoding="bytes")

    images = batch[b"data"].reshape(-1, 3, 32, 32)
    images = images.transpose(0, 2, 3, 1)
    labels = np.array(batch[b"labels"])

    return images, labels

# Load all five training batches
train_batches = [
    load_batch(os.path.join(DATA_DIR, f"data_batch_{i}"))
    for i in range(1, 6)
]

x_all = np.concatenate([batch[0] for batch in train_batches])
y_all = np.concatenate([batch[1] for batch in train_batches])

# Load the original test batch
x_test_first, y_test = load_batch(
    os.path.join(DATA_DIR, "test_batch")
)

y_all = y_all.flatten()
y_test = y_test.flatten()

print("Training images:", x_all.shape)
print("Training labels:", y_all.shape)
print("Test images:", x_test_first.shape)
print("Test labels:", y_test.shape)

Training images: (50000, 32, 32, 3)
Training labels: (50000,)
Test images: (10000, 32, 32, 3)
Test labels: (10000,)


In [71]:
# Splitting data in training and val 

def make_small_split(x, y, per_class_total=50, per_class_train=40, seed=42):
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []
    for c in range(10):
        idx = np.where(y == c)[0]                                  
        chosen = rng.choice(idx, per_class_total, replace=False)    
        train_idx.extend(chosen[:per_class_train])                  
        val_idx.extend(chosen[per_class_train:])                    
    return x[train_idx], y[train_idx], x[val_idx], y[val_idx]

x_train, y_train, x_val, y_val = make_small_split(x_all, y_all, seed=SEED)
print("train:", x_train.shape)
print("val  :", x_val.shape)

train: (400, 32, 32, 3)
val  : (100, 32, 32, 3)


In [72]:
# Shifted Test dataset 

import numpy as np
from scipy import ndimage

SEED = 42
rng = np.random.default_rng(SEED)

x_shifted = x_test_first.copy()
n = len(x_test_first)

# Shuffle indices but not repeat 
indices = rng.permutation(n)

# Divide 10,000 images into four groups of 2,500
group_size = n // 4
rotation_idx = indices[:group_size]
brightness_idx = indices[group_size:2*group_size]
occlusion_idx = indices[2*group_size:3*group_size]
hidden_idx = indices[3*group_size:]


for i in rotation_idx:
    angle = rng.uniform(-25, 25)
    x_shifted[i] = np.clip(
        ndimage.rotate(
            x_test_first[i].astype(np.float32),
            angle, axes=(0, 1), reshape=False,
            mode="reflect", order=1), 
            0, 255).astype(np.uint8)


for i in brightness_idx:
    img = x_test_first[i].astype(np.float32)
    contrast = rng.uniform(0.6, 1.4)
    brightness = rng.uniform(-40, 40)
    img = (img - 128) * contrast + 128 + brightness
    x_shifted[i] = np.clip(img, 0, 255).astype(np.uint8)


for i in occlusion_idx:
    img = x_test_first[i].copy()
    top, left = rng.integers(0, 17, size=2)
    img[top:top+16, left:left+16, :] = rng.integers(0, 256)
    x_shifted[i] = img


for i in hidden_idx:
    img = x_test_first[i].astype(np.float32)
    gray = (
        0.299 * img[:, :, 0]
        + 0.587 * img[:, :, 1]
        + 0.114 * img[:, :, 2]
    )
    x_shifted[i] = np.stack([gray, gray, gray], axis=-1).astype(np.uint8)


y_shifted = y_test.copy()

print("Clean test:", x_test_first.shape)
print("Shifted test:", x_shifted.shape)
print("Rotation images:", len(rotation_idx))
print("Brightness/contrast images:", len(brightness_idx))
print("Occlusion images:", len(occlusion_idx))
print("Hidden corruption images:", len(hidden_idx))



Clean test: (10000, 32, 32, 3)
Shifted test: (10000, 32, 32, 3)
Rotation images: 2500
Brightness/contrast images: 2500
Occlusion images: 2500
Hidden corruption images: 2500


In [73]:
# Normalize data
def to_float(x):
    return x.astype("float32") / 255.0

x_train = to_float(x_train)
x_val = to_float(x_val)
x_clean_test = to_float(x_test_first)
x_shifted_test = to_float(x_shifted)

In [74]:
                                                                   # Part 1 

In [75]:
def inception_block(x, filters):
    n1 = layers.Conv2D(filters, 1, padding="same", activation="relu")(x)

    n2 = layers.Conv2D(filters, 1, padding="same", activation="relu")(x)
    n2 = layers.Conv2D(filters, 3, padding="same", activation="relu")(n2)

    n3 = layers.Conv2D(filters, 1, padding="same", activation="relu")(x)
    n3 = layers.Conv2D(filters, 5, padding="same", activation="relu")(n3)

    n4 = layers.MaxPooling2D(pool_size=3, strides=1, padding="same")(x)
    n4 = layers.Conv2D(filters, 1, padding="same", activation="relu")(n4)

    return layers.Concatenate()(
        [n1, n2, n3, n4]
    )


In [76]:
# Follow Le-net style 
def build_model():

    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Conv2D(32, kernel_size=5, padding="same", activation="relu")(inputs)
    x = layers.AveragePooling2D(pool_size=2)(x)

    x = inception_block(x,16)

    x = layers.Conv2D( 64, kernel_size=5, padding="same", activation="relu")(x)
    x = layers.AveragePooling2D(pool_size=2)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)

    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    return model


In [77]:
model = build_model()
model.summary()

Model: "functional_38"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_38      │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_304 (Conv2D) │ (None, 32, 32,    │      2,432 │ input_layer_38[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ average_pooling2d_… │ (None, 16, 16,    │          0 │ conv2d_304[0][0]  │
│ (AveragePooling2D)  │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_306 (Conv2D) │ (None, 16, 16,    │        528 │ average_pooling2… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_308 (Conv2D) │ (None, 16, 16,    │        528 │ average_pooling2… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_38    │ (None, 16, 16,    │          0 │ average_pooling2… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_305 (Conv2D) │ (None, 16, 16,    │        528 │ average_pooling2… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_307 (Conv2D) │ (None, 16, 16,    │      2,320 │ conv2d_306[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_309 (Conv2D) │ (None, 16, 16,    │      6,416 │ conv2d_308[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_310 (Conv2D) │ (None, 16, 16,    │        528 │ max_pooling2d_38… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_38      │ (None, 16, 16,    │          0 │ conv2d_305[0][0], │
│ (Concatenate)       │ 64)               │            │ conv2d_307[0][0], │
│                     │                   │            │ conv2d_309[0][0], │
│                     │                   │            │ conv2d_310[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_311 (Conv2D) │ (None, 16, 16,    │    102,464 │ concatenate_38[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ average_pooling2d_… │ (None, 8, 8, 64)  │          0 │ conv2d_311[0][0]  │
│ (AveragePooling2D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_38          │ (None, 4096)      │          0 │ average_pooling2… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_76 (Dense)    │ (None, 128)       │    524,416 │ flatten_38[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_77 (Dense)    │ (None, 10)        │      1,290 │ dense_76[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 641,450 (2.45 MB)

 Trainable params: 641,450 (2.45 MB)

 Non-trainable params: 0 (0.00 B)

In [78]:
# Rotation
def rotation_aug(img, rng):
    angle = rng.uniform(-25, 25)
    img = ndimage.rotate(
        img.astype(np.float32),
        angle,axes=(0, 1),
        reshape=False,mode="reflect",order=1)
    return np.clip(img, 0, 255).astype(np.uint8)


# Brightness and contrast
def brightness_contrast_aug(img, rng):
    img = img.astype(np.float32)
    contrast = rng.uniform(0.6, 1.4)
    brightness = rng.uniform(-40, 40)
    img = (img - 128) * contrast + 128 + brightness
    return np.clip(img, 0, 255).astype(np.uint8)


# Random occlusion
def occlusion_aug(img, rng):
    img = img.copy()
    top, left = rng.integers(0, 17, size=2)
    img[top:top+16, left:left+16, :] = rng.integers(0, 256, size=(16, 16, 1), dtype=np.uint8)
    return img


# Grayscale
def grayscale_aug(img, rng):
    img = img.astype(np.float32)
    gray = (
        0.299 * img[:, :, 0]
        + 0.587 * img[:, :, 1]
        + 0.114 * img[:, :, 2]
    )
    gray = np.clip(gray, 0, 255).astype(np.uint8)
    return np.stack([gray, gray, gray], axis=-1)

In [79]:
def increase_dataset(x, y, aug_function, seed):
    rng = np.random.default_rng(seed)

    # Remove normalized 
    if x.dtype != np.uint8:
        x_raw = np.clip(x * 255, 0, 255).astype(np.uint8)
    else:
        x_raw = x.copy()

    x_aug = np.array([
        aug_function(img, rng) for img in x_raw
    ])

    # concatination of  originals and augmented copy 
    x_combined = np.concatenate([x_raw, x_aug], axis=0)
    y_combined = np.concatenate([y, y], axis=0)

    x_combined = x_combined.astype("float32") / 255.0

    return x_combined, y_combined

In [ ]:
rotation_result = []
brightness_contrast_result = []
occlusion_result = []
grayscale_result = []
non_augmentation = []

SEEDS = [42, 43, 44]

for i in SEEDS:

    keras.utils.set_random_seed(i)

    # No augmentation
    print("Result of non-augmentation:", i)

    model = build_model()
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    history = model.fit(
        x_train, y_train,
        epochs=60,
        batch_size=32,
        validation_data=(x_val, y_val),
    )
    
    print ("\nEvaluation Of results : \n")
    val_loss, val_acc = model.evaluate(x_val, y_val)
    clean_loss, clean_acc = model.evaluate(x_clean_test, y_test)
    shifted_loss, shifted_acc = model.evaluate(x_shifted_test, y_shifted)

    non_augmentation.append({
        "Seed": i,
        "Val Acc": val_acc,
        "Clean Test": clean_acc,
        "Shifted Test": shifted_acc
    })


    # Rotation
    print("Result of rotation:", i)

    x_rotation, y_rotation = increase_dataset(x_train, y_train, rotation_aug, seed=i)

    keras.utils.set_random_seed(i)
    model = build_model()
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    history = model.fit(
        x_rotation, y_rotation,
        epochs=60,
        batch_size=32,
        validation_data=(x_val, y_val),
    )
    print ("\nEvaluation Of results : \n")
    val_loss, val_acc = model.evaluate(x_val, y_val)
    clean_loss, clean_acc = model.evaluate(x_clean_test, y_test)
    shifted_loss, shifted_acc = model.evaluate(x_shifted_test, y_shifted)

    rotation_result.append({
        "Seed": i,
        "Val Acc": val_acc,
        "Clean Test": clean_acc,
        "Shifted Test": shifted_acc
    })


    # Brightness and contrast
    print("\nResult of brightness/contrast:", i)

    x_brightness, y_brightness = increase_dataset(x_train, y_train, brightness_contrast_aug, seed=i)

    keras.utils.set_random_seed(i)
    model = build_model()
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy", metrics=["accuracy"])


    history = model.fit(
        x_brightness, y_brightness,
        epochs=60,
        batch_size=32,
        validation_data=(x_val, y_val)
    )
    print ("\nEvaluation Of results : \n")
    val_loss, val_acc = model.evaluate(x_val, y_val)
    clean_loss, clean_acc = model.evaluate(x_clean_test, y_test)
    shifted_loss, shifted_acc = model.evaluate(x_shifted_test, y_shifted)

    brightness_contrast_result.append({
        "Seed": i,
        "Val Acc": val_acc,
        "Clean Test": clean_acc,
        "Shifted Test": shifted_acc
    })


    # Occlusion
    print("\nResult of occlusion:", i)

    x_occlusion, y_occlusion = increase_dataset(x_train, y_train, occlusion_aug, seed=i)

    keras.utils.set_random_seed(i)
    model = build_model()
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy", metrics=["accuracy"])


    history = model.fit(
        x_occlusion, y_occlusion,
        epochs=60,
        batch_size=32,
        validation_data=(x_val, y_val)
    )
    print ("\nEvaluation Of results : \n")
    val_loss, val_acc = model.evaluate(x_val, y_val)
    clean_loss, clean_acc = model.evaluate(x_clean_test, y_test)
    shifted_loss, shifted_acc = model.evaluate(x_shifted_test, y_shifted)

    occlusion_result.append({
        "Seed": i,
        "Val Acc": val_acc,
        "Clean Test": clean_acc,
        "Shifted Test": shifted_acc
    })


    # Grayscale
    print("\nResult of grayscale:", i)

    x_grayscale, y_grayscale = increase_dataset(x_train, y_train, grayscale_aug, seed=i)

    keras.utils.set_random_seed(i)
    model = build_model()
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy", metrics=["accuracy"])


    history = model.fit(
        x_grayscale, y_grayscale,
        epochs=60,
        batch_size=32,
        validation_data=(x_val, y_val)
    )
    print ("\nEvaluation Of results : \n")
    val_loss, val_acc = model.evaluate(x_val, y_val)
    clean_loss, clean_acc = model.evaluate(x_clean_test, y_test)
    shifted_loss, shifted_acc = model.evaluate(x_shifted_test, y_shifted)

    grayscale_result.append({
        "Seed": i,
        "Val Acc": val_acc,
        "Clean Test": clean_acc,
        "Shifted Test": shifted_acc
    })




Result of non-augmentation: 42
Epoch 1/60
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 294ms/step - accuracy: 0.0975 - loss: 2.3061 - val_accuracy: 0.1100 - val_loss: 2.2842
Epoch 2/60
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1375 - loss: 2.2466 - val_accuracy: 0.2100 - val_loss: 2.1345
Epoch 3/60
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2175 - loss: 2.1061 - val_accuracy: 0.2200 - val_loss: 2.0519
Epoch 4/60
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2825 - loss: 1.9998 - val_accuracy: 0.2600 - val_loss: 1.9809
Epoch 5/60
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3400 - loss: 1.8767 - val_accuracy: 0.2900 - val_loss: 1.9689
Epoch 6/60
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4275 - loss: 1.6795 - val_accuracy: 0.3900 - val_loss: 1.9563
Epoch 7/60
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4775 - loss: 1.5366 - val_accuracy: 0.2900 - val_loss: 1.9853
Epoch 8/60
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4875 - loss: 1

In [ ]:
# Follow LeNet style
def build_model2():

    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Conv2D(32, kernel_size=5, padding="same", use_bias=False, kernel_regularizer=regularizers.l2(0.0001))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.AveragePooling2D(pool_size=2)(x)

    x = inception_block(x,16)

    x = layers.Conv2D(64, kernel_size=5, padding="same", use_bias=False, kernel_regularizer=regularizers.l2(0.0001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.AveragePooling2D(pool_size=2)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.0001))(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(10, activation="softmax")(x)

    model2 = keras.Model(inputs, outputs)
    return model2

In [ ]:
# Start with the original training dataset
x_train_all = x_train.copy()
y_train_all = y_train.copy()

# Generate two augmented copies for each augmentation type
x_rotation1, y_rotation1 = increase_dataset(
    x_train, y_train, rotation_aug, seed=42
)

x_rotation2, y_rotation2 = increase_dataset(
    x_train, y_train, rotation_aug, seed=43
)

x_brightness1, y_brightness1 = increase_dataset(
    x_train, y_train, brightness_contrast_aug, seed=44
)

x_brightness2, y_brightness2 = increase_dataset(
    x_train, y_train, brightness_contrast_aug, seed=45
)

x_occlusion1, y_occlusion1 = increase_dataset(
    x_train, y_train, occlusion_aug, seed=46
)

x_occlusion2, y_occlusion2 = increase_dataset(
    x_train, y_train, occlusion_aug, seed=47
)

x_grayscale1, y_grayscale1 = increase_dataset(
    x_train, y_train, grayscale_aug, seed=48
)

x_grayscale2, y_grayscale2 = increase_dataset(
    x_train, y_train, grayscale_aug, seed=49
)

# Combine original images and augmented copies
x_final = np.concatenate([
    x_train,
    x_rotation1[400:],
    x_rotation2[400:],
    x_brightness1[400:],
    x_brightness2[400:],
    x_occlusion1[400:],
    x_occlusion2[400:],
    x_grayscale1[400:],
    x_grayscale2[400:]
], axis=0)

y_final = np.concatenate([
    y_train,
    y_rotation1[400:],
    y_rotation2[400:],
    y_brightness1[400:],
    y_brightness2[400:],
    y_occlusion1[400:],
    y_occlusion2[400:],
    y_grayscale1[400:],
    y_grayscale2[400:]
], axis=0)

print("Training images:", len(x_final))
print("Training labels:", len(y_final))

In [ ]:
final_model=[]

for i in SEEDS:

    keras.utils.set_random_seed(i)
    model2 = build_model2()
    
    model2.compile(optimizer="adam",loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    
    history2 = model2.fit(
        x_final,
        y_final,
        epochs=60,
        batch_size=32,
        validation_data=(x_val, y_val),
        shuffle=True,
    )
    print ("\nEvaluation Of results : \n")
    val_loss, val_acc = model2.evaluate(x_val, y_val)
    clean_loss, clean_acc = model2.evaluate(x_clean_test, y_test)
    shifted_loss, shifted_acc = model2.evaluate(x_shifted_test, y_shifted)

    final_model.append({
        "Seed": i,
        "Val Acc": val_acc,
        "Clean Test": clean_acc,
        "Shifted Test": shifted_acc
    })

In [ ]:
results_df = pd.concat([
    pd.DataFrame(non_augmentation).assign(Configuration="No augmentation"),
    pd.DataFrame(rotation_result).assign(Configuration="Rotation"),
    pd.DataFrame(brightness_contrast_result).assign(Configuration="Brightness/Contrast"),
    pd.DataFrame(occlusion_result).assign(Configuration="Occlusion"),
    pd.DataFrame(grayscale_result).assign(Configuration="Grayscale"),
    pd.DataFrame(final_model).assign(Configuration="Updated Model")
], ignore_index=True)

display(results_df)

In [ ]:
print(results_df.columns)

In [ ]:
summary = results_df.groupby("Configuration").agg({
    "Val Acc": ["mean", "std"],
    "Clean Test": ["mean", "std"],
    "Shifted Test": ["mean", "std"]
})

summary.columns = [
    "Val Acc Mean", "Val Acc Std",
    "Clean Test Mean", "Clean Test Std",
    "Shifted Test Mean", "Shifted Test Std"
]

display(summary)

In [ ]:
# Part 5

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

y_pred = np.argmax(model2.predict(x_clean_test), axis=1)

class_names = ["airplane", "automobile", "bird", "cat", "deer","dog", "frog", "horse", "ship", "truck"]

print(classification_report(
    y_test, y_pred,
    target_names=class_names,
    digits=3,
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, normalize="true")

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names
)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Normalized Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
cm_counts = confusion_matrix(y_test, y_pred)
np.fill_diagonal(cm_counts, 0)

top_confusions = []

for i in range(10):
    for j in range(10):
        if cm_counts[i, j] > 0:
            top_confusions.append(
                (class_names[i], class_names[j], cm_counts[i, j])
            )

top_confusions.sort(key=lambda x: x[2], reverse=True)   # use 3rd value for sorting

print("Top 5 class confusions:")
for actual, predicted, count in top_confusions[:5]:
    print(f"{actual} predicted as {predicted}: {count} images")

In [ ]:
# The model confuses airplanes with ships because CIFAR-10 images are small, making their shapes and details difficult to distinguish. Similar backgrounds, such as blue sky and water, may also confuse the model.

In [ ]:
wrong_predict = np.where(y_pred != y_test)[0]

plt.figure(figsize=(12, 6))

for i, idx in enumerate(wrong_predict[:8]):
    plt.subplot(2, 4, i + 1)
    plt.imshow(x_clean_test[idx])
    plt.title(
        f"Actual: {class_names[y_test[idx]]}\n"
        f"Predicted: {class_names[y_pred[idx]]}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
noise = rng.normal(0, 0.1, x_clean_test.shape)

x_hidden_test = np.clip(x_clean_test + noise, 0, 1)

hidden_loss, hidden_acc = model2.evaluate(x_hidden_test, y_test)

print(f"Hidden-corruption accuracy: {hidden_acc:.2%}")

In [ ]:
clean_loss, clean_acc = model2.evaluate(x_clean_test, y_test)

print(f"Clean test accuracy: {clean_acc:.2%}")
print(f"Hidden-corruption accuracy: {hidden_acc:.2%}")


In [ ]:
# the result on gussian noise and clean dataset is almost same so we can say that our model is correctly generalized 

In [ ]:
# Part F

In [ ]:
import numpy as np
import time

start = time.perf_counter()
pred_normal = model2.predict(x_shifted_test)
normal_time = time.perf_counter() - start
normal_pred = np.argmax(pred_normal, axis=1)
normal_acc = np.mean(normal_pred == y_shifted)


start = time.perf_counter()
pred_original = model2.predict(x_shifted_test)
pred_flipped = model2.predict(np.flip(x_shifted_test, axis=2))      # horizantal shift as axis = 2 is width shift
avg_probs = (pred_original + pred_flipped) / 2   

tta_pred = np.argmax(avg_probs, axis=1)
tta_acc = np.mean(tta_pred == y_shifted)

tta_time = time.perf_counter() - start

# Results
print(f"Normal accuracy: {normal_acc:.2%}")
print(f"TTA accuracy: {tta_acc:.2%}")
print(f"Accuracy change: {(tta_acc - normal_acc) * 100:.2f} percentage points")
print(f"Normal inference time: {normal_time:.2f} seconds")
print(f"TTA inference time: {tta_time:.2f} seconds")